<a href="https://colab.research.google.com/github/shubham-senani/Marketing-Analytics/blob/main/MA14_Price_Analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Price Analytics

### Loading Data

In [ ]:
# ==========================================
# INSTALL REQUIRED LIBRARIES (Colab only)
# ==========================================


# ==========================================
# IMPORT LIBRARIES
# ==========================================
import pandas as pd
import numpy as np
import re
import io
from google.colab import files
import statsmodels.api as sm

In [ ]:
def load_dataset(google_sheet_id=None):
    """
    Loads dataset either by:
    1 → Uploading a CSV file manually
    2 → Loading a Google Sheet (if ID is provided)

    Parameters:
    google_sheet_id (str, optional): Google Sheet ID.
                                     If provided, automatically loads Sheet.

    Returns:
    pd.DataFrame
    """

    # -----------------------------------
    # IF GOOGLE SHEET ID IS PROVIDED → AUTO LOAD (Option 2)
    # -----------------------------------
    if google_sheet_id is not None:
        data_url = f"https://docs.google.com/uc?id={google_sheet_id}&export=download"
        df = pd.read_csv(data_url)
        print("Google Sheet loaded successfully.")

    else:
        # Ask user only if ID not provided
        print("Choose how you want to provide the dataset:")
        print("1 → Upload CSV file (from your computer)")
        print("2 → Load Google Sheet (as CSV)")

        choice = input("Enter 1 or 2: ").strip()

        # OPTION 1: Upload CSV
        if choice == "1":
            uploaded = files.upload()
            filename = list(uploaded.keys())[0]
            df = pd.read_csv(io.BytesIO(uploaded[filename]))
            print(f"File '{filename}' uploaded successfully.")

        # OPTION 2: Ask for Google Sheet ID
        elif choice == "2":
            google_sheet_id = input("Enter the Google Sheet ID: ").strip()
            data_url = f"https://docs.google.com/uc?id={google_sheet_id}&export=download"
            df = pd.read_csv(data_url)
            print("Google Sheet loaded successfully.")

        else:
            raise ValueError("Invalid choice. Please rerun and enter 1 or 2.")

    # -----------------------------------
    # DATA CHECK
    # -----------------------------------
    print("\nDataset shape:", df.shape)
    print("Columns:", df.columns.tolist())

    return df

## Log-Log Modelling

### Building the Model

In [ ]:
# ==========================================
# LOAD DATASETS
# ==========================================
# Provide Google Sheet IDs OR load manually

sku_180 = load_dataset("1WT-JL1fazmT8wBKKAbqwL_gbhGSk4CNV")
sku_350 = load_dataset("1I37xpmiDTeSbfdVTXtn-rlxpbTQPE0Yt")
sku_700 = load_dataset("1Y9plIKdFhKG8sWWELN-OGy53mA_ERB1L")

Google Sheet loaded successfully.

Dataset shape: (32, 8)
Columns: ['Year', 'Month', 'Month-Year', 'price', 'volume', 'inflation_rate', 'gdp_yoy', 'trade_budget_in_USD']
Google Sheet loaded successfully.

Dataset shape: (32, 8)
Columns: ['Year', 'Month', 'Month-Year', 'price', 'volume', 'inflation_rate', 'gdp_yoy', 'trade_budget_in_USD']
Google Sheet loaded successfully.

Dataset shape: (32, 8)
Columns: ['Year', 'Month', 'Month-Year', 'price', 'volume', 'inflation_rate', 'gdp_yoy', 'trade_budget_in_USD']


In [ ]:
# ==========================================
# FEATURE ENGINEERING FUNCTION
# (Equivalent to your inflation adjustment + log transforms)
# ==========================================

def prepare_dataset(df):
    """
    Creates:
    - Inflation adjusted price
    - Log volume
    - Log adjusted price
    - Log trade promotion (+1 to avoid log(0))
    """

    df = df.copy()

    # Inflation adjusted price
    df["adj_price"] = df["price"] / (1 + df["inflation_rate"])

    # Log transformations (log-log model)
    df["logVolume"] = np.log(df["volume"])
    df["log_adj_price"] = np.log(df["adj_price"])
    df["log_trade_promotion"] = np.log(df["trade_budget_in_USD"] + 1)

    return df


# Apply to all SKUs
sku_180 = prepare_dataset(sku_180)
sku_350 = prepare_dataset(sku_350)
sku_700 = prepare_dataset(sku_700)

In [ ]:
# ==========================================
# STEP 3: REGRESSION FUNCTION
# (Equivalent to lm() in R)
# ==========================================

def run_regression(df, sku_name="SKU"):
    """
    Runs log-log regression:
    logVolume ~ gdp_yoy + log_adj_price + log_trade_promotion
    """

    X = df[["gdp_yoy", "log_adj_price", "log_trade_promotion"]]
    X = sm.add_constant(X)   # Adds intercept

    y = df["logVolume"]

    model = sm.OLS(y, X).fit()

    print(f"\n==============================")
    print(f"Regression Results: {sku_name}")
    print(f"==============================")
    print(model.summary())

    return model

### Analyzing the Models

In [ ]:
model_180 = run_regression(sku_180, "SKU 180")


Regression Results: SKU 180
                            OLS Regression Results                            
Dep. Variable:              logVolume   R-squared:                       0.107
Model:                            OLS   Adj. R-squared:                  0.011
Method:                 Least Squares   F-statistic:                     1.114
Date:                Tue, 03 Mar 2026   Prob (F-statistic):              0.360
Time:                        07:54:48   Log-Likelihood:                 38.909
No. Observations:                  32   AIC:                            -69.82
Df Residuals:                      28   BIC:                            -63.96
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
const

In [ ]:
model_350 = run_regression(sku_350, "SKU 350")


Regression Results: SKU 350
                            OLS Regression Results                            
Dep. Variable:              logVolume   R-squared:                       0.207
Model:                            OLS   Adj. R-squared:                  0.122
Method:                 Least Squares   F-statistic:                     2.437
Date:                Tue, 03 Mar 2026   Prob (F-statistic):             0.0855
Time:                        07:54:52   Log-Likelihood:                 48.231
No. Observations:                  32   AIC:                            -88.46
Df Residuals:                      28   BIC:                            -82.60
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
const

In [ ]:
model_700 = run_regression(sku_700, "SKU 700")


Regression Results: SKU 700
                            OLS Regression Results                            
Dep. Variable:              logVolume   R-squared:                       0.011
Model:                            OLS   Adj. R-squared:                 -0.095
Method:                 Least Squares   F-statistic:                   0.09991
Date:                Tue, 03 Mar 2026   Prob (F-statistic):              0.959
Time:                        07:54:54   Log-Likelihood:                 36.543
No. Observations:                  32   AIC:                            -65.09
Df Residuals:                      28   BIC:                            -59.22
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
const